# Train microduck_rl on Colab

**Purpose:** viability test for running training on a Colab GPU you've already
paid for, as an alternative to [Hugging Face Jobs](../hf/README.md) billing.

This is a manual runbook, not automation — unlike `--hf-jobs`, there's no API
to submit this headlessly, and no persistent container: you drive it cell by
cell from an open browser tab. Only worth it if the smoke test below actually
passes on whatever GPU Colab hands you.

Before running: **Runtime > Change runtime type > GPU** (whichever tier your
plan gives you — T4 on free, L4/A100 on Pro/Pro+).

## 1. Confirm a GPU actually got attached

In [ ]:
!nvidia-smi

## 2. Clone the repo

`microduck_rl` is private, so this needs a GitHub token with `repo` scope.
Add it once as a Colab secret named `GITHUB_TOKEN` (key icon in the left
sidebar) rather than pasting it into a cell — secrets aren't saved into the
notebook file and aren't shown in cell output.

In [ ]:
import os

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    from getpass import getpass
    github_token = getpass("GitHub token (repo scope, private clone): ")

os.environ["GITHUB_TOKEN"] = github_token

# credentials go through an env var, not a Python f-string into the URL,
# so they never land in cell input/output history.
!git clone --branch feat/hop-env-training \
    "https://${GITHUB_TOKEN}@github.com/pollen-robotics/microduck_rl.git"

In [ ]:
%cd /content/microduck_rl

## 3. Install `uv` and sync deps

Pinned to Python 3.12 (see `pyproject.toml`) — `uv` will fetch that
interpreter itself if Colab's default doesn't match. First sync pulls ~2 GB
of CUDA wheels, same as the ARM-box note in the main README — give it a long
HTTP timeout.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version

In [ ]:
import os
os.environ["UV_HTTP_TIMEOUT"] = "600"
!uv sync

## 4. Headless MuJoCo rendering (EGL)

Same fix already needed for HF Jobs (see `hf_jobs.py` git history): the
container has no display, `mujoco.Renderer()` needs EGL explicitly selected,
and the base image doesn't ship `libEGL.so.1`. Colab's GPU runtime is in the
same boat — do this even if you don't plan to use `--video`, since some
tasks touch the renderer regardless.

In [ ]:
!apt-get -qq update && apt-get -qq install -y libegl1
import os
os.environ["MUJOCO_GL"] = "egl"

## 5. wandb login

Logs this run into the same wandb project as the HF Jobs runs, so it shows
up next to them and can be resumed the same way
(`--agent.resume True --agent.load-checkpoint ... --wandb-run-path ...`).

In [ ]:
import os

try:
    from google.colab import userdata
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    from getpass import getpass
    wandb_key = getpass("wandb API key: ")

os.environ["WANDB_API_KEY"] = wandb_key
!uv run wandb login --relogin "$WANDB_API_KEY"

## 6. Smoke test

Small env count, ~1 minute of iterations. The point is only to prove MuJoCo
Warp + CUDA torch actually run on this GPU — not to produce a useful
checkpoint. **Don't skip to §7 before this passes.**

In [ ]:
!uv run train Mjlab-Velocity-Flat-MicroDuck \
    --env.scene.num-envs 256 \
    --agent.max-iterations 50 \
    --agent.run-name colab-smoke-test

## 7. A real run, once the smoke test is clean

Scale `num-envs`/`max-iterations` up to whatever this GPU tier can hold.
Colab has no equivalent to HF Jobs' `--detach` — the process dies if the
runtime disconnects, so keep the tab open (or use Colab Pro's background
execution) and lean on the wandb-checkpoint resume path if it drops.

Defaults below target the hop task (both the butt-bounce and no-crawl/worming
fixes are on this branch, per commits `6484dcc`/`3a99fd7` — see
`docs/ideas/hop-behavior.md` and the reward functions in `mdp.py` if you want
the detail). Edit `TASK` / `NUM_ENVS` / `MAX_ITERATIONS` / `RUN_NAME` below and
run.

In [ ]:
TASK = "Mjlab-Hop-Flat-MicroDuck"
NUM_ENVS = 4096
MAX_ITERATIONS = 6000
RUN_NAME = "colab-hop-1"

!uv run train {TASK} \
    --env.scene.num-envs {NUM_ENVS} \
    --agent.max-iterations {MAX_ITERATIONS} \
    --agent.run-name {RUN_NAME}

### Resuming after a dropped session

```python
!uv run train {TASK} \
    --env.scene.num-envs {NUM_ENVS} \
    --agent.max-iterations {MAX_ITERATIONS} \
    --agent.run-name {RUN_NAME}-resume \
    --agent.resume True \
    --wandb-run-path <entity/project/run_id>
```

Note the same gotcha as HF Jobs resumes: `--agent.max-iterations` is *how
many more iterations from here*, not an absolute target — if you want to
stop at 6000 total and you resumed at iteration 4250, pass `1750`, not
`6000`.